# keybed segmentation training

Trains `KeybedSegNet` on the baked keybed corpus and writes `keybed_seg.pt` to the notebook output.

Corpus comes from `make lab-corpus-zip`: `frames/*.png` at the net's input size plus `corners.json` of normalised quads.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path
import json
import math

import cv2
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

# kaggle decides how deep it nests an uploaded archive, so the corpus is located by its
# manifest rather than by a mount path we would have to keep in step with the upload
CORPUS = next(Path("/kaggle/input").rglob("corners.json")).parent
OUT = Path("/kaggle/working/keybed_seg.pt")

INPUT_SIZE = 288
MASK_SIZE = 144
# masks are drawn large and area-downsampled so the target edge is soft the same way
# the local pipeline's is; rasterising straight at 144 would train against hard edges
RASTER_SIZE = MASK_SIZE * 4
VAL_FRACTION = 0.08
EPOCHS = 40
BATCH_PER_GPU = 128
LEARNING_RATE = 2e-3
MIN_LEARNING_RATE = 1e-5
SEED = 0

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpus = torch.cuda.device_count()
batch_size = BATCH_PER_GPU * max(gpus, 1)
print(f"corpus {CORPUS}")
print(f"{gpus} gpu(s), batch {batch_size}")

In [ ]:
corners = json.loads((CORPUS / "corners.json").read_text())
stems = sorted(corners)


def load(stem):
    image = cv2.imread(str(CORPUS / "frames" / f"{stem}.png"), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return None
    quad = np.array(corners[stem], dtype=np.float64) * RASTER_SIZE
    big = np.zeros((RASTER_SIZE, RASTER_SIZE), dtype=np.uint8)
    cv2.fillPoly(big, [quad.astype(np.int32)], 255)
    mask = cv2.resize(big, (MASK_SIZE, MASK_SIZE), interpolation=cv2.INTER_AREA)
    return image, mask


with ThreadPoolExecutor(max_workers=8) as workers:
    pairs = [p for p in workers.map(load, stems) if p is not None]

images = np.stack([p[0] for p in pairs]).astype(np.float32) / 255.0
masks = np.stack([p[1] for p in pairs]).astype(np.float32) / 255.0
print(f"{len(images)} frames  images {images.shape}  masks {masks.shape}")

In [ ]:
rng = np.random.default_rng(SEED)
order = rng.permutation(len(images))
cut = int(len(order) * VAL_FRACTION)
val_index, train_index = order[:cut], order[cut:]

# the corpus is fixed, so photometric jitter is what keeps forty epochs from memorising it;
# nothing geometric, a flipped keybed would break the corner convention the labels carry
def augment(batch, generator):
    count = batch.shape[0]
    gain = generator.uniform(0.6, 1.5, size=(count, 1, 1)).astype(np.float32)
    bias = generator.uniform(-0.2, 0.2, size=(count, 1, 1)).astype(np.float32)
    gamma = generator.uniform(0.7, 1.4, size=(count, 1, 1)).astype(np.float32)
    out = np.clip(batch, 1e-4, 1.0) ** gamma * gain + bias
    out += generator.normal(0.0, 0.03, size=batch.shape).astype(np.float32)
    return np.clip(out, 0.0, 1.0)


val_images = torch.from_numpy(images[val_index]).unsqueeze(1)
val_masks = torch.from_numpy(masks[val_index]).unsqueeze(1)
print(f"train {len(train_index)}  val {len(val_index)}")

In [ ]:
class KeybedSegNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(
            nn.Conv2d(1, 16, 3, stride=2, padding=1),
            nn.GroupNorm(4, 16),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, stride=2, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.Conv2d(32, 64, 3, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.Conv2d(64, 96, 3, stride=2, padding=1),
            nn.GroupNorm(8, 96),
            nn.ReLU(),
        )
        self.head = nn.Sequential(
            nn.ConvTranspose2d(96, 64, 4, stride=2, padding=1),
            nn.GroupNorm(8, 64),
            nn.ReLU(),
            nn.ConvTranspose2d(64, 32, 4, stride=2, padding=1),
            nn.GroupNorm(8, 32),
            nn.ReLU(),
            nn.ConvTranspose2d(32, 16, 4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 1, 1),
        )

    def forward(self, x):
        return self.head(self.body(x))


torch.manual_seed(SEED)
model = KeybedSegNet().to(device)
print(sum(p.numel() for p in model.parameters()), "params")
parallel = nn.DataParallel(model) if gpus > 1 else model

In [ ]:
@torch.no_grad()
def validate():
    model.eval()
    intersection = union = 0.0
    for start in range(0, len(val_images), batch_size):
        inputs = val_images[start : start + batch_size].to(device)
        truth = val_masks[start : start + batch_size].to(device) > 0.5
        predicted = torch.sigmoid(model(inputs)) > 0.5
        intersection += float((predicted & truth).sum())
        union += float((predicted | truth).sum())
    return intersection / max(union, 1.0)


optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE)
steps = math.ceil(len(train_index) / batch_size) * EPOCHS
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=steps, eta_min=MIN_LEARNING_RATE
)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

best = 0.0
for epoch in range(EPOCHS):
    parallel.train()
    shuffled = rng.permutation(train_index)
    for start in range(0, len(shuffled), batch_size):
        pick = shuffled[start : start + batch_size]
        inputs = torch.from_numpy(augment(images[pick], rng)).unsqueeze(1).to(device)
        truth = torch.from_numpy(masks[pick]).unsqueeze(1).to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            loss = F.binary_cross_entropy_with_logits(parallel(inputs), truth)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
    iou = validate()
    marker = ""
    if iou > best:
        best = iou
        marker = " *"
        torch.save(model.state_dict(), OUT)
    print(f"epoch {epoch + 1}/{EPOCHS} val_iou {iou:.4f}{marker}", flush=True)

print(f"saved {OUT} val_iou {best:.4f}")